<a href="https://colab.research.google.com/github/young-tryler/ThucHanhDeepLearning/blob/main/2001230980_BuiQuocTri_B2_THDL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 2001230980 - Bùi Quốc Trí  - Lab2

In [ ]:
#LAB 2 LinearRegression

from sklearn import preprocessing
# Install dependencies as needed:
# !pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "weatherHistory.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,      # Muốn trả về dữ liệu dưới dạng pandas.core.frame.DataFrame
  "budincsevity/szeged-weather",    # Tên người đăng / tên dataset . Nó thấy trên link khi truy cập vào dataset https://www.kaggle.com/datasets/budincsevity/szeged-weather
  file_path,                        # Tên của file chưa dataset trên kanggle
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

# print(type(df))
# print("First 5 records:", df.head())

print("KÍCH THƯỚC DATASET: ", df.shape)
# print(df.info())

Y = df['Temperature (C)']
X = df[['Apparent Temperature (C)','Humidity','Wind Speed (km/h)','Wind Bearing (degrees)','Visibility (km)','Loud Cover','Pressure (millibars)','Precip Type']]
X['Precip Type'] = X['Precip Type'].fillna('unknown')

from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42) # 30% test 70% training

# print(f"Kích thước X_train:{X_train.shape}")
# print(f"Kích thước Y_train:{Y_train.shape}")
# print(f"Kích thước X_test:{X_test.shape}")
# print(f"Kích thước Y_test:{Y_test.shape}")

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

num_features = ['Apparent Temperature (C)','Humidity','Wind Speed (km/h)','Wind Bearing (degrees)','Visibility (km)','Loud Cover','Pressure (millibars)']
cat_features = ['Precip Type']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features)
    ])
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed = preprocessor.transform(X_test)

from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_processed, Y_train)

# for col, coef in zip(X.columns, model.coef_):
#   print(f"{col} : {coef}")

from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

Y_pred = model.predict(X_test_processed)

mse = mean_squared_error(Y_test, Y_pred)

rmse = np.sqrt(mse)

r2 = r2_score(Y_test, Y_pred)

print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'R2: {r2}')

import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

# Trích xuất 60 điểm dữ liệu liên tiếp từ tập test để vẽ biểu đồ sắc nét, dễ quan sát
plt.plot(Y_test.values[:60], label='Nhiệt độ Thực tế (Actual)', color='teal', marker='o', alpha=0.8)
plt.plot(Y_pred[:60], label='Nhiệt độ Dự đoán (Predicted)', color='crimson', linestyle='--', marker='x', alpha=0.9)

plt.title('Biểu đồ so sánh kết quả: Nhiệt độ thực tế vs Mô hình dự đoán (60 mẫu)', fontsize=14)
plt.xlabel('Chỉ số mẫu thử nghiệm', fontsize=12)
plt.ylabel('Nhiệt độ (°C)', fontsize=12)
plt.legend(fontsize=11)
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()


In [ ]:
#LAB 2 CNN
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt
import numpy as np

# ==========================================
# 1. TẢI VÀ TIỀN XỬ LÝ DỮ LIỆU MỚI (FASHION MNIST)
# ==========================================
# Thay vì gán mnist, ta tải tập dữ liệu thời trang fashion_mnist
(X_train, y_train), (X_test, y_test) = datasets.fashion_mnist.load_data()

# Danh sách 10 nhãn tương ứng với các giá trị từ 0 đến 9 trong tập dữ liệu mới
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

# Chuẩn hóa giá trị pixel từ [0, 255] về đoạn [0, 1] kiểu số thực float32
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Định dạng lại kích thước ảnh (Reshape) chuẩn 1 kênh màu (ảnh xám) để đưa vào CNN
X_train = X_train.reshape((X_train.shape[0], 28, 28, 1))
X_test = X_test.reshape((X_test.shape[0], 28, 28, 1))

print("Kích thước tập Train mới:", X_train.shape)
print("Kích thước tập Test mới:", X_test.shape)

# ==========================================
# 2. XÂY DỰNG KIẾN TRÚC MÔ HÌNH CNN
# ==========================================
model = models.Sequential()

# Tầng Chập 1: 32 bộ lọc 3x3, kích hoạt relu, nhận đầu vào ảnh thời trang 28x28x1
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))

# Tầng Chập 2: 64 bộ lọc 3x3, kích hoạt relu
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))

# Tầng Chập 3: 64 bộ lọc 3x3, kích hoạt relu
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

# Tầng Duỗi phẳng & tầng kết nối đầy đủ (Dense)
model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))

# Tầng phân loại đầu ra: 10 nơ-ron tương ứng với 10 phân lớp trang phục, kích hoạt softmax
model.add(layers.Dense(10, activation='softmax'))

print("\n--- Cấu trúc mô hình CNN cho dữ liệu Thời trang ---")
model.summary()

# ==========================================
# 3. BIÊN DỊCH VÀ HUẤN LUYỆN MÔ HÌNH
# ==========================================
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("\n--- Bắt đầu huấn luyện với tập dữ liệu mới ---")
model_fit = model.fit(X_train, y_train,
                      epochs=5,
                      batch_size=64,
                      validation_data=(X_test, y_test))

# ==========================================
# 4. ĐÁNH GIÁ MÔ HÌNH VÀ DỰ BÁO TRỰC QUAN
# ==========================================
print("\n--- Đánh giá mô hình trên tập Test mới ---")
score = model.evaluate(X_test, y_test, verbose=2)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

# Dự báo thử nghiệm nhãn của tấm ảnh mẫu đầu tiên trong tập Test
img_sample = X_test[0]
img_sample_batch = np.expand_dims(img_sample, axis=0)

predictions = model.predict(img_sample_batch)
predicted_label_index = np.argmax(predictions)

print(f"\nNhãn số dự báo: {predicted_label_index} -> Tên sản phẩm: {class_names[predicted_label_index]}")
print(f"Nhãn thực tế gốc: {y_test[0]} -> Tên sản phẩm gốc: {class_names[y_test[0]]}")
# ==========================================
# 4. TRỰC QUAN HÓA ĐỒ THỊ LOSS / ACCURACY (BỔ SUNG THEO ĐÚNG TÀI LIỆU)
# ==========================================
plt.figure(figsize=(12, 4))

# Vẽ đồ thị Accuracy (Độ chính xác)
plt.subplot(1, 2, 1)
plt.plot(model_fit.history['accuracy'], label='Train Accuracy')
plt.plot(model_fit.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Val'], loc='upper left')

# Vẽ đồ thị Loss (Hàm mất mát)
plt.subplot(1, 2, 2)
plt.plot(model_fit.history['loss'], label='Train Loss')
plt.plot(model_fit.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Val'], loc='upper left')

plt.show()

# ==========================================
# 5. ĐÁNH GIÁ MÔ HÌNH VÀ DỰ BÁO CHI TIẾT
# ==========================================
print("\n--- Đánh giá mô hình trên tập Test mới ---")
score = model.evaluate(X_test, y_test, verbose=2)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

# Dự báo thử nghiệm nhãn của tấm ảnh mẫu đầu tiên trong tập Test
img_sample = X_test[0]

# Bản chất NumPy: Chuyển ma trận ảnh (28, 28, 1) thành Batch dữ liệu (1, 28, 28, 1)
img_sample_batch = np.expand_dims(img_sample, axis=0)

predictions = model.predict(img_sample_batch)
predicted_label_index = np.argmax(predictions)

print(f"\nNhãn số dự báo: {predicted_label_index} -> Tên sản phẩm: {class_names[predicted_label_index]}")
print(f"Nhãn thực tế gốc: {y_test[0]} -> Tên sản phẩm gốc: {class_names[y_test[0]]}")

# ==========================================
# 6. LƯU THAM SỐ MÔ HÌNH (MỤC 2.10 TRONG FILE PDF)
# ==========================================
model.save('cnn_fashion_mnist_model.h5')
print("\nĐã lưu cấu trúc và trọng số mô hình thành công vào file 'cnn_fashion_mnist_model.h5'")